# Generate Atomic Data Reference for TARDIS
This noteboook generates atomic data to be compared with the [atomic data stored in the tardis regression data repository](https://github.com/tardis-sn/tardis-regression-data/blob/main/atom_data/kurucz_cd23_chianti_H_He.h5) for testing purposes. 

** As of May 14, 2025, the Chianti database exported by Carsus is NOT compatible with versions of TARDIS before v2025.5.18. Use a version of Carsus before this data to generate computed Chianti data suitable for older versions of TARDIS. **


## `kurucz_cd23_chianti_H_He_latest.h5` reproduction status

The TARDIS regression file `kurucz_cd23_chianti_H_He_latest.h5` uses the legacy TARDIS atom-data HDF schema (`database_version='v0.9'`), Kurucz GFALL levels with label-aware level identity, CHIANTI H-He collisions, Knox-Long zeta data, NNDC decay radiation data, and Astropy constants pinned to the CODATA 2010 / IAU 2012 set before importing `astropy.units` or `astropy.constants`.

The matching historical Kurucz source is `linelists/gfall/gfall.dat` in `tardis-sn/carsus-data-kurucz` (`sha256=e4b46b8e7d1460536754e345970d090659c7bdc3f796a8bdfc637d45aa94e42e`). `GFALLReader` reads 564949 raw rows and produces 563247 parsed rows. Its O IV rows preserve both lower-side and upper-side assignments for the `2p3 *2D` term (`255155.900 cm-1` with `J=1.5` and `J=2.5`, and `255184.900 cm-1` with `J=0.5` and `J=1.5`), which reproduces the four reference O IV levels near 31.635 eV. The flat file regenerates the classic SQLite `gfall` table content, so the SQLite database and reader are not needed for reconstruction.

The matching NNDC decay radiation source is the `csv/` snapshot from `tardis-sn/carsus-data-nndc` commit `136a8633e3dee21079d3738dabfe7758f6e40d41` (`ENSDF CSV conversions`). Parsed with `NNDCReader`, this produces `/decay_radiation_data` shape `(242295, 41)`, matching the reference schema and row index. The local `/home/afullard/Downloads/tardisnuclear/decay_radiation.h5` file is not the reference source; it contains `/decay_radiation` with shape `(95, 9)`.

The reference CHIANTI data are from the CHIANTI 7.x family, not the current local CHIANTI 10.0 tree. Local version checks found `/home/afullard/chianti` = `10.0`, `/home/afullard/chianti_nine` = `9.0.1`, `/home/afullard/chianti_seven` = `7.1`, and `/storage/cloudy/data/chianti` = `7.1.4`. The reference contains CHIANTI 7.x H I, He I, and He II radiative transition values, including six ultra-long-wavelength He I lines, indexed `(2, 0, 22, 26)`, `(2, 0, 22, 27)`, `(2, 0, 23, 26)`, `(2, 0, 23, 28)`, `(2, 0, 24, 28)`, and `(2, 0, 25, 29)`, that are present in CHIANTI 7-style `he_1.wgfa` data and absent from the CHIANTI 10.0 reader output. Current ChiantiPy 0.16 cannot directly parse the local CHIANTI 7.1 tree because of old `elvlc` formatting, so reproduction with current software uses CHIANTI 10.0 for parseable levels/collisions and overlays the CHIANTI 7.1 H-He `.wgfa` radiative records by level index. A verified candidate generated this way matches the reference table keys, row counts, row indexes, `/atom_data`, `/levels_data`, `/ionization_data`, `/zeta_data`, `/collisions_metadata`, and `/macro_atom_references`; remaining byte-level differences are metadata/signature values, NNDC float/string serialization, collision energy rounding, and small H-He line-ID shifts from legacy pre-filter numbering.


In [ ]:
# Settings used for the legacy TARDIS H-He reference atom-data layout
from pathlib import Path

import pandas as pd

from carsus.io.kurucz import GFALLReader
from carsus.io.chianti_ import ChiantiReader
from carsus.io.nuclear import NNDCReader

CHIANTI_7_ROOT = Path('/home/afullard/chianti_seven')  # VERSION -> 7.1


def read_chianti_7_wgfa(path, atomic_number, ion_charge):
    rows = []
    with path.open() as wgfa_file:
        for line in wgfa_file:
            parts = line.split()
            if not parts:
                continue
            if parts[0] == '-1':
                break
            rows.append(
                (
                    atomic_number,
                    ion_charge,
                    int(parts[0]) - 1,
                    int(parts[1]) - 1,
                    float(parts[2]) / 10.0,
                    float(parts[3]),
                    float(parts[4]),
                )
            )
    return rows


def overlay_chianti_7_h_he_lines(chianti_reader):
    rows = []
    rows.extend(read_chianti_7_wgfa(CHIANTI_7_ROOT / 'h/h_1/h_1.wgfa', 1, 0))
    rows.extend(read_chianti_7_wgfa(CHIANTI_7_ROOT / 'he/he_1/he_1.wgfa', 2, 0))
    rows.extend(read_chianti_7_wgfa(CHIANTI_7_ROOT / 'he/he_2/he_2.wgfa', 2, 1))
    chianti_7_lines = pd.DataFrame(
        rows,
        columns=[
            'atomic_number',
            'ion_charge',
            'level_index_lower',
            'level_index_upper',
            'wavelength',
            'gf',
            'A_ul',
        ],
    )
    string_dtype = pd.StringDtype(na_value=pd.NA)
    for column in ['energy_upper', 'j_upper', 'energy_lower', 'j_lower']:
        chianti_7_lines[column] = pd.Series(
            pd.NA, index=chianti_7_lines.index, dtype=string_dtype
        )
    chianti_7_lines = chianti_7_lines.set_index(
        ['atomic_number', 'ion_charge', 'level_index_lower', 'level_index_upper']
    )
    chianti_7_lines = chianti_7_lines.sort_index()
    chianti_7_lines = chianti_7_lines[chianti_reader.lines.columns]
    other_lines = chianti_reader.lines.drop(chianti_7_lines.index, errors='ignore')
    chianti_reader.lines = pd.concat([other_lines, chianti_7_lines], sort=True)

gfall_reader = GFALLReader(
    'H-Zn',
    unique_level_identifier=['energy', 'j', 'label'],
)

# The reference CHIANTI data are CHIANTI 7.x. Local evidence:
#   /home/afullard/chianti_seven/VERSION -> 7.1
#   /storage/cloudy/data/chianti/VERSION -> 7.1.4
#   /home/afullard/chianti/VERSION -> 10.0, which has revised H-He wgfa data.
# Current ChiantiPy 0.16 cannot directly parse the local CHIANTI 7.1 elvlc
# files, so use the current reader and overlay CHIANTI 7.1 H-He wgfa rows.
chianti_reader = ChiantiReader('H-He', collisions=True, priority=20)
overlay_chianti_7_h_he_lines(chianti_reader)

# Exact decay reproduction uses tardis-sn/carsus-data-nndc commit:
# 136a8633e3dee21079d3738dabfe7758f6e40d41 (ENSDF CSV conversions)
nndc_reader = NNDCReader(dirname='/tmp/carsus-data-nndc-history/csv')

# After constructing TARDISAtomData, write the legacy-compatible schema with:
# atom_data.to_hdf(
#     'kurucz_cd23_chianti_H_He_latest.h5',
#     legacy_tardis_schema=True,
#     database_version='v0.9',
# )


In [ ]:
import pathlib
from carsus.io.nist import NISTWeightsComp, NISTIonizationEnergies

In [ ]:
atomic_weights = NISTWeightsComp()
ionization_energies = NISTIonizationEnergies('H-Zn', )

In [ ]:
from carsus.io.kurucz import GFALLReader

gfall_reader = GFALLReader('H-Zn')

In [ ]:
cmfgen_path = '../../carsus-data-cmfgen/atomic/'
if not pathlib.Path(cmfgen_path).exists():
    cmfgen_path = "/tmp/atomic/"

In [ ]:
from carsus.io.cmfgen import CMFGENReader

cmfgen_reader = CMFGENReader.from_config('Si 0-1',
                                         cmfgen_path,
                                         priority=30,
                                         ionization_energies=True,
                                         cross_sections=True,
                                         collisions=True,
                                         temperature_grid=None,
                                         drop_mismatched_labels=True)


In [ ]:
from carsus.io.zeta import KnoxLongZeta

zeta_data = KnoxLongZeta()

In [ ]:
from carsus.io.output import TARDISAtomData

atom_data = TARDISAtomData(atomic_weights,
                           ionization_energies,
                           gfall_reader,
                           zeta_data,
                           cmfgen_reader=cmfgen_reader)

In [ ]:
atom_data.to_hdf('kurucz_cd23_cmfgen_H_Si.h5')